# Kỹ thuật 3 — Hough đa tỉ lệ

**Phần 1** là cài đặt: tự viết, KHÔNG dùng AI. **Phần 2** là thí nghiệm: có dùng AI, đã khai báo prompt ở ô markdown trên từng ô mã.

Quy ước: ảnh lưu dạng `I[y, x]`. `grad_dir` là mảng `(H, W)` chứa θg ∈ [0, π) tại mỗi điểm biên, ρ = x·cosθ + y·sinθ.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

# PHẦN 1 — CÀI ĐẶT (tự viết, không dùng AI)

In [ ]:
# Dán convolve2d của nhóm từ Phần A vào đây (find_peaks phải dùng hàm này để làm mượt A)
# def convolve2d(...): ...

def hough_accumulate(edges, theta_range, rho_range, n_theta, n_rho,
                     grad_dir=None, delta_deg=None):
    """Bỏ phiếu vào mảng tích lũy. Nếu grad_dir khác None, chỉ bỏ phiếu
    trong dải +/- delta_deg quanh hướng gradient.
    Trả về (A, thetas, rhos)."""
    raise NotImplementedError


def find_peaks(A, n_peaks, nms_radius=3, smooth_sigma=1.0):
    """Làm mượt A bằng convolve2d của nhóm, triệt phi cực đại,
    trả về danh sách (rho, theta, votes) sắp giảm dần."""
    raise NotImplementedError


def hough_multiscale(edges, grad_dir, n_levels=2, refine_factor=8, n_peaks=3):
    """Hough thô-đến-mịn. Trả về danh sách (rho, theta) đã tinh chỉnh.
    Nhớ bẫy quấn vòng ở theta = 0 / pi (đổi dấu rho)."""
    raise NotImplementedError

# PHẦN 2 — THÍ NGHIỆM MINH HỌA

## 2.0 Hàm phụ trợ: ảnh tổng hợp có đáp án, đo sai số, vẽ hình

### Có dùng AI hỗ trợ
Công cụ: Claude (Anthropic)
Mục đích: sinh ảnh tổng hợp có đáp án (rho0, theta0) đúng, hàm đo sai số góc/rho, hàm vẽ

Prompt:
> (DÁN NGUYÊN VĂN prompt bạn đã gửi vào đây)

Đã sửa lại những gì: (ghi ngắn gọn phần bạn phải sửa để chạy đúng)

In [ ]:
def make_synthetic(shape=(400, 600), segments=(), pos_noise=0.5,
                   n_junk=300, dir_noise_deg=3.0, seed=0):
    """segments: list (theta0_deg, cx, cy, length) - đường đi qua (cx, cy).
    Trả về edges (H,W) uint8, grad_dir (H,W) float64 in [0,pi), gt [(rho0, theta0_rad)]."""
    rng = np.random.default_rng(seed)
    H, W = shape
    edges = np.zeros((H, W), dtype=np.uint8)
    grad_dir = np.zeros((H, W), dtype=np.float64)
    gt = []
    for theta_deg, cx, cy, length in segments:
        th = np.deg2rad(theta_deg)
        gt.append((cx*np.cos(th) + cy*np.sin(th), th))
        t = np.arange(-length/2, length/2, 0.5)
        nrm = rng.normal(0, pos_noise, size=t.shape)      # nhiễu vị trí theo pháp tuyến
        x = cx + t*(-np.sin(th)) + nrm*np.cos(th)
        y = cy + t*np.cos(th) + nrm*np.sin(th)
        xi, yi = np.round(x).astype(int), np.round(y).astype(int)
        ok = (xi >= 0) & (xi < W) & (yi >= 0) & (yi < H)
        edges[yi[ok], xi[ok]] = 255
        grad_dir[yi[ok], xi[ok]] = (th + np.deg2rad(rng.normal(0, dir_noise_deg, ok.sum()))) % np.pi
    jy, jx = rng.integers(0, H, n_junk), rng.integers(0, W, n_junk)   # điểm biên rác
    edges[jy, jx] = 255
    grad_dir[jy, jx] = rng.uniform(0, np.pi, n_junk)
    return edges, grad_dir, gt


def line_diff(est, gt):
    """(sai số góc [độ], sai số rho [px]); xử lý (rho, theta) == (-rho, theta ± pi)."""
    r, t = est
    r0, t0 = gt
    best = None
    for rr, tt in ((r, t), (-r, t - np.pi), (-r, t + np.pi)):
        d = (abs(tt - t0), abs(rr - r0))
        if best is None or d[0] < best[0]:
            best = d
    return np.rad2deg(best[0]), best[1]


def evaluate(detected, gt, max_deg=5.0, max_rho=10.0):
    """Với mỗi đường đúng lấy đường phát hiện gần nhất; None nếu bị bỏ sót."""
    out = []
    for g in gt:
        cands = [line_diff(d, g) for d in detected]
        if not cands:
            out.append(None); continue
        best = min(cands, key=lambda d: d[0] + d[1])
        out.append(best if (best[0] <= max_deg and best[1] <= max_rho) else None)
    return out


def run_standard(edges, d_theta_deg, d_rho, grad_dir=None, delta_deg=None, n_peaks=3):
    """Hough chuẩn tắc 1 độ phân giải. Trả về (lines, A, peaks, thời gian giây)."""
    H, W = edges.shape
    D = np.hypot(H, W)
    n_theta, n_rho = int(round(180/d_theta_deg)), int(round(2*D/d_rho))
    t0 = time.perf_counter()
    A, thetas, rhos = hough_accumulate(edges, (0, np.pi), (-D, D), n_theta, n_rho,
                                       grad_dir=grad_dir, delta_deg=delta_deg)
    peaks = find_peaks(A, n_peaks)
    dt = time.perf_counter() - t0
    return [(p[0], p[1]) for p in peaks], A, peaks, dt


def run_multiscale(edges, grad_dir, n_levels=2, r=8, n_peaks=3):
    t0 = time.perf_counter()
    lines = hough_multiscale(edges, grad_dir, n_levels=n_levels, refine_factor=r, n_peaks=n_peaks)
    return list(lines), None, None, time.perf_counter() - t0


def peak_to_background(A, peaks):
    """Tỉ số: số phiếu trung bình của các đỉnh / số phiếu trung bình của ô có phiếu."""
    if A is None or not peaks:
        return np.nan
    A = A.astype(np.float64)
    return np.mean([p[2] for p in peaks]) / A[A > 0].mean()


def draw_lines(ax, shape, lines, color='r', lw=1.5):
    H, W = shape
    for rho, theta in lines:
        a, b = np.cos(theta), np.sin(theta)
        x0, y0 = a*rho, b*rho
        ax.plot([x0 - 2000*b, x0 + 2000*b], [y0 + 2000*a, y0 - 2000*a], color=color, lw=lw)
    ax.set_xlim(0, W); ax.set_ylim(H, 0)


# Bộ đường thẳng chuẩn: 1 đoạn rất dài, 1 đoạn vừa, 1 đoạn rất ngắn
TEST_SEGMENTS = [(70, 300, 200, 500), (20, 450, 250, 150), (130, 120, 320, 30)]
N_PEAKS = 3

## 2.a-1 Ảnh hưởng của Δθ và Δρ (mảng tích lũy + đường phát hiện)

### Có dùng AI hỗ trợ
Công cụ: Claude (Anthropic)
Mục đích: dựng vòng lặp quét Δθ, Δρ và vẽ lưới mảng tích lũy

Prompt:
> (DÁN NGUYÊN VĂN prompt bạn đã gửi vào đây)

Đã sửa lại những gì: (ghi ngắn gọn phần bạn phải sửa để chạy đúng)

In [ ]:
edges, gdir, gt = make_synthetic(segments=TEST_SEGMENTS, seed=1)

def sweep(name, values, fixed):
    fig, axes = plt.subplots(2, len(values), figsize=(4.5*len(values), 7))
    for j, v in enumerate(values):
        dth, drho = (v, fixed) if name == 'dtheta' else (fixed, v)
        lines, A, peaks, dt = run_standard(edges, dth, drho, n_peaks=N_PEAKS)
        axes[0, j].imshow(A, aspect='auto', cmap='magma')
        axes[0, j].set_title(f'dθ={dth}°, dρ={drho}px | max={A.max()}')
        axes[1, j].imshow(edges, cmap='gray', interpolation='nearest')
        draw_lines(axes[1, j], edges.shape, lines)
        axes[1, j].set_title(f'{dt:.2f}s')
    fig.suptitle(f'Quét {name}'); plt.tight_layout(); plt.show()

sweep('dtheta', [0.25, 0.5, 1, 2], fixed=1)    # Δρ cố định = 1px
sweep('drho',   [0.5, 1, 2, 4],   fixed=1)     # Δθ cố định = 1°

## 2.a-2 Sai số góc theo số tầng n và hệ số tinh chỉnh r

### Có dùng AI hỗ trợ
Công cụ: Claude (Anthropic)
Mục đích: quét n_levels và refine_factor, vẽ sai số góc theo n (thang log) cùng đường lý thuyết Δθ0/r^n

Prompt:
> (DÁN NGUYÊN VĂN prompt bạn đã gửi vào đây)

Đã sửa lại những gì: (ghi ngắn gọn phần bạn phải sửa để chạy đúng)

In [ ]:
def mean_angle_err(n_levels, r, seeds=(0, 1, 2)):
    errs = []
    for s in seeds:
        e, g, gt_ = make_synthetic(segments=TEST_SEGMENTS, seed=s)
        lines, _, _, _ = run_multiscale(e, g, n_levels=n_levels, r=r, n_peaks=N_PEAKS)
        errs += [x[0] for x in evaluate(lines, gt_) if x is not None]
    return np.mean(errs) if errs else np.nan

N_LEVELS = [1, 2, 3, 4]
R_FACTORS = [2, 4, 8, 16]
plt.figure(figsize=(7, 5))
for r in R_FACTORS:
    plt.semilogy(N_LEVELS, [mean_angle_err(n, r) for n in N_LEVELS], 'o-', label=f'r={r}')
    plt.semilogy(N_LEVELS, [1.0 / r**n for n in N_LEVELS], '--', alpha=0.3, color='gray')  # lý thuyết, Δθ0 = 1°
plt.xlabel('số tầng n'); plt.ylabel('sai số góc trung bình (độ)')
plt.title('Sai số góc theo n (nét đứt: Δθ0 / r^n)'); plt.legend(); plt.grid(True, which='both', alpha=0.3)
plt.show()

## 2.b So sánh: Hough thô / mịn / đa tỉ lệ (+ bật/tắt bỏ phiếu theo gradient)

### Có dùng AI hỗ trợ
Công cụ: Claude (Anthropic)
Mục đích: dựng bộ ảnh so sánh và bảng ba chỉ số: sai số góc, tỉ số đỉnh/nền, thời gian

Prompt:
> (DÁN NGUYÊN VĂN prompt bạn đã gửi vào đây)

Đã sửa lại những gì: (ghi ngắn gọn phần bạn phải sửa để chạy đúng)

In [ ]:
DELTA = 10  # độ, dung sai hướng gradient
IMAGES = {
    'hỗn hợp (dài/vừa/ngắn)': make_synthetic(segments=TEST_SEGMENTS, seed=1),
    'nhiễu nhiều':            make_synthetic(segments=TEST_SEGMENTS, pos_noise=1.0, n_junk=3000, seed=2),
    'gần thẳng đứng':         make_synthetic(segments=[(0.5, 200, 200, 350), (179.5, 400, 200, 350)], seed=3),
}
METHODS = {
    'thô 1°/2px':            lambda e, g: run_standard(e, 1, 2, n_peaks=N_PEAKS),
    'mịn 0.1°/0.5px':        lambda e, g: run_standard(e, 0.1, 0.5, n_peaks=N_PEAKS),
    'thô + gradient':        lambda e, g: run_standard(e, 1, 2, grad_dir=g, delta_deg=DELTA, n_peaks=N_PEAKS),
    'mịn + gradient':        lambda e, g: run_standard(e, 0.1, 0.5, grad_dir=g, delta_deg=DELTA, n_peaks=N_PEAKS),
    'đa tỉ lệ (gradient)':   lambda e, g: run_multiscale(e, g, n_levels=2, r=8, n_peaks=N_PEAKS),
}

for img_name, (e, g, gt_) in IMAGES.items():
    print(f'\n=== {img_name} ===')
    print(f'{"phương pháp":<22}{"sai số góc TB":>14}{"bỏ sót":>8}{"đỉnh/nền":>10}{"thời gian":>11}')
    for m_name, fn in METHODS.items():
        lines, A, peaks, dt = fn(e, g)
        res = evaluate(lines, gt_)
        ok = [x[0] for x in res if x is not None]
        err = f'{np.mean(ok):.3f}°' if ok else '   —'
        print(f'{m_name:<22}{err:>14}{len(res)-len(ok):>8}{peak_to_background(A, peaks):>10.2f}{dt:>10.2f}s')

## 2.c Hai trường hợp bắt buộc: gần thẳng đứng & dài + ngắn

### Có dùng AI hỗ trợ
Công cụ: Claude (Anthropic)
Mục đích: dựng hai ảnh bắt buộc, bài kiểm thử quấn vòng θ=0.5° / 179.5° và hình so sánh

Prompt:
> (DÁN NGUYÊN VĂN prompt bạn đã gửi vào đây)

Đã sửa lại những gì: (ghi ngắn gọn phần bạn phải sửa để chạy đúng)

In [ ]:
# Kiểm thử quấn vòng riêng
for th in (0.5, 179.5):
    e, g, gt_ = make_synthetic(segments=[(th, 300, 200, 350)], n_junk=100, seed=5)
    lines, _, _, _ = run_multiscale(e, g, n_levels=2, r=8, n_peaks=N_PEAKS)
    r_ = evaluate(lines, gt_)[0]
    print(f'theta0={th}°:', 'PASS' if (r_ is not None and r_[0] < 0.2) else 'FAIL', r_)

CASES = {
    'gần thẳng đứng': make_synthetic(segments=[(0.5, 200, 200, 350), (179.5, 400, 200, 350)], seed=3),
    'dài 550px + ngắn 20px': make_synthetic(segments=[(70, 300, 200, 550), (130, 120, 320, 20)], seed=4),
}
RUNNERS = {
    'thô 1°/2px': lambda e, g: run_standard(e, 1, 2, n_peaks=N_PEAKS),
    'mịn 0.1°/0.5px': lambda e, g: run_standard(e, 0.1, 0.5, n_peaks=N_PEAKS),
    'đa tỉ lệ': lambda e, g: run_multiscale(e, g, n_levels=2, r=8, n_peaks=N_PEAKS),
}
fig, axes = plt.subplots(len(CASES), len(RUNNERS), figsize=(5*len(RUNNERS), 4*len(CASES)))
for i, (c_name, (e, g, gt_)) in enumerate(CASES.items()):
    for j, (m_name, fn) in enumerate(RUNNERS.items()):
        lines, _, _, _ = fn(e, g)
        res = evaluate(lines, gt_)
        axes[i, j].imshow(e, cmap='gray', interpolation='nearest')
        draw_lines(axes[i, j], e.shape, [(r, t) for r, t in zip([x[0] for x in gt_], [x[1] for x in gt_])], color='lime', lw=0.8)
        draw_lines(axes[i, j], e.shape, lines, color='r', lw=1)
        axes[i, j].set_title(f'{c_name}\n{m_name}: bỏ sót {sum(x is None for x in res)}/{len(res)}')
plt.tight_layout(); plt.show()

## 2.d Kiểm lại trên ảnh thật

### Có dùng AI hỗ trợ
Công cụ: Claude (Anthropic)
Mục đích: đọc ảnh thật, Canny + Sobel lấy hướng gradient, chạy Hough đa tỉ lệ và vẽ kết quả

Prompt:
> (DÁN NGUYÊN VĂN prompt bạn đã gửi vào đây)

Đã sửa lại những gì: (ghi ngắn gọn phần bạn phải sửa để chạy đúng)

In [ ]:
import cv2
try:
    img = cv2.imread('road2.jpg')
    gray = cv2.GaussianBlur(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), (5, 5), 1.4)
    edges_r = cv2.Canny(gray, 100, 200)
    gx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    gdir_r = np.arctan2(gy, gx) % np.pi          # pháp tuyến của đường thẳng
    lines, _, _, dt = run_multiscale(edges_r, gdir_r, n_levels=2, r=8, n_peaks=8)
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    draw_lines(ax, img.shape[:2], lines)
    ax.set_title(f'Hough đa tỉ lệ - {dt:.2f}s'); ax.axis('off'); plt.show()
except Exception as ex:
    print('Chưa chạy được ảnh thật:', ex)

## 2.e Nhận xét (tự viết)

Phần này do bạn tự điền sau khi chạy. Gợi ý các câu cần trả lời:
- Δθ mịn dần thì mảng tích lũy thay đổi thế nào, đoạn dài / đoạn ngắn hỏng ra sao?
- Sai số góc theo n có bám đường Δθ0/rⁿ không? Từ n nào thì ngừng cải thiện, vì sao?
- Bỏ phiếu theo gradient cải thiện tỉ số đỉnh/nền và thời gian bao nhiêu lần?
- Hough một tỉ lệ hỏng ở ảnh nào trong 2.c, còn đa tỉ lệ có sửa được không?